In [ ]:
# Cell 1 — Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('All libraries imported successfully.')

In [ ]:
# Cell 2 — Load Dataset
df = pd.read_csv('Churn_Modelling.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Cell 3 — Dataset Shape, Info & Descriptive Statistics
print('── Shape ──────────────────────────────')
print(df.shape)

print('\n── Data Types & Non-Null Counts ────────')
df.info()

print('\n── Descriptive Statistics ──────────────')
df.describe()

In [ ]:
# Cell 4 — Check Missing Values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing >= 0].to_string())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Cell 5 — Visualize Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df['Exited'].value_counts()
axes[0].bar(['Retained (0)', 'Churned (1)'], churn_counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Churn Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Customer Churn Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Correlation Heatmap
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['RowNumber', 'CustomerId'])

plt.figure(figsize=(12, 8))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Data Preprocessing

# 7a. Drop irrelevant columns
df_clean = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
print('Dropped: RowNumber, CustomerId, Surname')

# 7b. Label Encode Gender
le = LabelEncoder()
df_clean['Gender'] = le.fit_transform(df_clean['Gender'])  # Female=0, Male=1
print('Label encoded: Gender (Female=0, Male=1)')

# 7c. One-Hot Encode Geography
df_clean = pd.get_dummies(df_clean, columns=['Geography'], drop_first=True)
print(f'One-hot encoded: Geography → {[c for c in df_clean.columns if "Geography" in c]}')

print(f'\nFinal shape after preprocessing: {df_clean.shape}')
df_clean.head()

In [ ]:
# Cell 8 — Feature / Target Split & Train-Test Split
X = df_clean.drop(columns=['Exited'])
y = df_clean['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print(f'Features     : {X_train.shape[1]}')

In [ ]:
# Cell 9 — Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('StandardScaler applied: fitted on training set, transformed both sets.')

In [ ]:
# Cell 10 — Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_acc   = accuracy_score(y_test, lr_preds)

print(f'Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')

In [ ]:
# Cell 11 — Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10,
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

rf_preds = rf_model.predict(X_test_scaled)
rf_acc   = accuracy_score(y_test, rf_preds)

print(f'Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')

In [ ]:
# Cell 12 — Model Evaluation: Logistic Regression
print('══════════════════════════════════════════')
print('       LOGISTIC REGRESSION EVALUATION     ')
print('══════════════════════════════════════════')
print(f'Accuracy : {lr_acc:.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_test, lr_preds))
print('\nClassification Report:')
print(classification_report(y_test, lr_preds, target_names=['Retained', 'Churned']))

# Confusion matrix heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, lr_preds), annot=True, fmt='d',
            cmap='Blues', xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            linewidths=1, linecolor='white')
plt.title('Logistic Regression — Confusion Matrix', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 — Model Evaluation: Random Forest
print('══════════════════════════════════════════')
print('       RANDOM FOREST EVALUATION           ')
print('══════════════════════════════════════════')
print(f'Accuracy : {rf_acc:.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_test, rf_preds))
print('\nClassification Report:')
print(classification_report(y_test, rf_preds, target_names=['Retained', 'Churned']))

# Confusion matrix heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, rf_preds), annot=True, fmt='d',
            cmap='Oranges', xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            linewidths=1, linecolor='white')
plt.title('Random Forest — Confusion Matrix', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Model Comparison Summary
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [round(lr_acc, 4), round(rf_acc, 4)]
})
comparison['Accuracy (%)'] = (comparison['Accuracy'] * 100).round(2)
print(comparison.to_string(index=False))

best_model = comparison.loc[comparison['Accuracy'].idxmax(), 'Model']
print(f'\n✅ Best performing model: {best_model}')

In [ ]:
# Cell 15 — Feature Importance Table (Random Forest)
feature_names = X.columns.tolist()
importances   = rf_model.feature_importances_

feat_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feat_df['Importance (%)'] = (feat_df['Importance'] * 100).round(2)
print('Feature Importance — Random Forest:')
print(feat_df.to_string(index=False))

In [ ]:
# Cell 16 — Feature Importance Bar Chart
plt.figure(figsize=(10, 6))
colors = sns.color_palette('viridis', len(feat_df))
bars = plt.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1],
                color=colors[::-1], edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, feat_df['Importance'][::-1]):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=9)

plt.title('Random Forest — Feature Importances', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.xlim(0, feat_df['Importance'].max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 17 — Final Conclusion
top_n = 5
top_features = feat_df.head(top_n)

print('=' * 55)
print('        CUSTOMER CHURN PREDICTION — SUMMARY        ')
print('=' * 55)

print(f"""
DATASET
  • Total records  : {df.shape[0]:,}
  • Features used  : {X.shape[1]}
  • Churn rate     : {y.mean()*100:.1f}%

MODEL PERFORMANCE
  • Logistic Regression  Accuracy : {lr_acc*100:.2f}%
  • Random Forest        Accuracy : {rf_acc*100:.2f}%  ← Best

TOP {top_n} FEATURES DRIVING CHURN (Random Forest):""")

for rank, row in enumerate(top_features.itertuples(), start=1):
    print(f'  {rank}. {row.Feature:<25} {row._3:.2f}%')

print("""
KEY INSIGHTS
  • Age is typically the strongest predictor — older
    customers churn more frequently.
  • NumOfProducts and IsActiveMember are strong
    behavioral indicators of retention.
  • Balance and EstimatedSalary reflect financial
    engagement with the bank.
  • Geography (Germany) shows notably higher churn rates.

RECOMMENDATION
  Random Forest outperforms Logistic Regression on this
  dataset. Focus retention campaigns on older, inactive
  customers — especially those in Germany with single
  products and high balances.
""")
print('=' * 55)